In [2]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler

if "__file__" in globals():
    project_root = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
else:
    project_root = os.path.abspath("..")

sys.path.append(project_root)

from ReusableFunctions.DataPreprocessing import DataPreprocessing
from ReusableFunctions.EvaluationMetrics import EvaluationMetrics as EM
from reproducibility_settings import set_global_seed

# Reproducibility
set_global_seed(seed=42, framework='numpy')


def train_and_evaluate_model(X_train, y_train, X_test, y_test, scaler_y, forecast_window):
    # Flatten for sklearn LR
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_test_flat = X_test.reshape(X_test.shape[0], -1)

    model = LinearRegression(fit_intercept=True)
    model.fit(X_train_flat, y_train)

    # Predictions
    train_preds_scaled = model.predict(X_train_flat)
    test_preds_scaled = model.predict(X_test_flat)

    train_preds_unscaled = scaler_y.inverse_transform(train_preds_scaled.reshape(-1, 1))
    test_preds_unscaled = scaler_y.inverse_transform(test_preds_scaled.reshape(-1, 1))

    y_train_true_unscaled = scaler_y.inverse_transform(y_train.reshape(-1, 1))
    y_test_true_unscaled = scaler_y.inverse_transform(y_test.reshape(-1, 1))

    # Metrics
    rmse = EM.rmse(y_test_true_unscaled, test_preds_unscaled)
    mape = EM.mape(y_test_true_unscaled, test_preds_unscaled)
    r2 = EM.r2(y_test_true_unscaled, test_preds_unscaled)
    acc = EM.accuracy(y_test_true_unscaled, test_preds_unscaled)
    profit_index = EM.profitability_index(y_test_true_unscaled, test_preds_unscaled, forecast_window)

    # Print metrics
    print("\n=== Evaluation Metrics ===")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAPE: {mape:.4f}")
    print(f"R2: {r2:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Profitability Index: {profit_index:.4f}")

    # Plot results
    plt.figure(figsize=(12, 5))
    plt.plot(y_test_true_unscaled, label='Actual')
    plt.plot(test_preds_unscaled, label='Predicted')
    plt.title("Predicted vs Actual Prices on Test Set (Linear Regression)")
    plt.xlabel("Time")
    plt.ylabel("Price")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return rmse, mape, r2, acc, profit_index


def split_dataset(X, y, train_ratio=0.8):
    split_index = int(len(X) * train_ratio)
    return X[:split_index], X[split_index:], y[:split_index], y[split_index:]


if __name__ == '__main__':
    ticker = 'AAPL'
    os.makedirs('check', exist_ok=True)

    # Fixed technical indicator combination
    fixed_indicator_combo = ['RSI', 'Upper_BB', 'Lower_BB', 'CCI', 'Williams_%R', 'OBV']

    window_size, forecast_window = 60, 30

    data_processor = DataPreprocessing(ticker=ticker, start_date='2014-01-01', end_date='2024-12-31')
    df = data_processor.add_technical_indicators()

    selected_features = ['Close'] + fixed_indicator_combo
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(df[selected_features])
    X, y = data_processor.create_windowed_data(scaled_data, window_size, forecast_window)
    X_train, X_test, y_train, y_test = split_dataset(X, y, train_ratio=0.8)

    # Target scaling
    scaler_y = MinMaxScaler()
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1))
    y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1))

    print(f"\nRunning Linear Regression for Fixed Indicators: {fixed_indicator_combo}, "
          f"Window Size: {window_size}, Forecast: {forecast_window}")

    train_and_evaluate_model(
        X_train, y_train_scaled,
        X_test, y_test_scaled,
        scaler_y, forecast_window
    )


ModuleNotFoundError: No module named 'torch'